In [1]:
#Imports
import sys
import os

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)


import pandas as pd
import numpy as np
from scipy.linalg import block_diag
from scipy.stats import chi2

from model_funcs import run_em, sort_matrix, create_Y

# Bayesian Mixture Model

**Goal**: Model the uncertainty expressed by multiple annotations
**Tool**: Multinomial Mixture Model
- There exists a latent ground truth, i.e. a "true label" for each image
- This latent ground truth is unambiguous but the annotator’s opinion about it is not
- We set up a distributional framework and model the observed annotations under the assumption that there exists a latent true label
- The parameters of the associated distributions allow us to analyse the sources of labelling uncertainty and we can estimate the latent labels through the posterior probabilities

Assume latent ground truth:

$Z^{(i)}\sim Multi(\boldsymbol{\pi}, 1)$ iid.  $\rightarrow$ prior

Given latent true class, the labellers' votes follow:

$Y^{(i)}|Z^{(i)}\sim Multi(\boldsymbol{\theta_{Z^{(i)}}}, J^i)$ iid. $\rightarrow$ likelihood

or explicitly:

$Y^{(i)}|(Z^{(i)} = airplane) \sim Multi(\boldsymbol{\theta_{airplane}}, J^i) \\
Y^{(i)}|(Z^{(i)} = automobile) \sim Multi(\boldsymbol{\theta_{automobile}}, J^i)\\
Y^{(i)}|(Z^{(i)} = bird) \sim Multi(\boldsymbol{\theta_{bird}}, J^i)\\
Y^{(i)}|(Z^{(i)} = cat) \sim Multi(\boldsymbol{\theta_{cat}}, J^i)\\
Y^{(i)}|(Z^{(i)} = deer) \sim Multi(\boldsymbol{\theta_{deer}}, J^i)\\
Y^{(i)}|(Z^{(i)} = dog) \sim Multi(\boldsymbol{\theta_{dog}}, J^i)\\
Y^{(i)}|(Z^{(i)} = frog) \sim Multi(\boldsymbol{\theta_{frog}}, J^i)\\
Y^{(i)}|(Z^{(i)} = horse) \sim Multi(\boldsymbol{\theta_{horse}}, J^i)\\
Y^{(i)}|(Z^{(i)} = ship) \sim Multi(\boldsymbol{\theta_{ship}}, J^i)\\
Y^{(i)}|(Z^{(i)} = truck) \sim Multi(\boldsymbol{\theta_{truck}}, J^i).$


Use Bayes Rule to model latent ground truth given votes:

$\tau^{(i)}_l = P(Z^{(i)}=l|Y^{(i)}) = \frac{P(Z^{(i)}=l) \cdot P(Y^{(i)}|Z^{(i)} = l)}{P(Y^{(i)})} = \frac{prior \cdot likelihood}{evidence}$



Apply Expectation Maximization (EM) algorithm, iteratively estimate latent ground truth and parameters

1. E-Step:
calculate posterior class probabilities $\tau^{(i)}_l$ given $\pi$ and $\theta_{Z^{(i)}}$, i.e., parameters of prior and likelihood
choose class with highest posterior as latent ground truth class $Z^{(i)}$

2. M-Step:
update $\pi$ and $\theta_{Z^{(i)}}$ given $\tau^{(i)}_l$, i.e.,  posterior class probabilities

Initial values:
- $\pi$ = (1/10, 1/10, 1/10, 1/10, 1/10, 1/10, 1/10, 1/10, 1/10, 1/10)
- $\theta$ drawn from dirichlet with $\alpha$ = (20,20,20,20,20,20,20,20,20,20) (i.e., 2* number_of_observed_classes)

# 1. Run SEM

In [73]:
# load data
cifar10h = pd.read_csv('../data/cifar10h.csv', index_col=0)

# draw a random sample 50% of annotator_ids from the cifar10h dataset and use it to create a split of the data
np.random.seed(0)
sample = np.random.choice(cifar10h.annotator_id.unique(), int(len(cifar10h.annotator_id.unique()) * 0.5), replace=False)
cifar10h_delta = cifar10h[cifar10h.annotator_id.isin(sample)]
cifar10h_theta = cifar10h[~cifar10h.annotator_id.isin(sample)]

In [103]:
rt_quantiles = np.quantile(cifar10h['reaction_time'].dropna(), q=np.arange(0.1, 1.0, 0.1))

rt_quantiles

array([1.044, 1.156, 1.256, 1.362, 1.483, 1.63 , 1.831, 2.157, 2.879])

In [ ]:
def run_em_for_quantile(q, data):
    # run EM for a single quantile
    R = data.loc[data['reaction_time'] > q].copy()
    R_Y = create_Y(R)
    R_one_hot = np.array(R_Y[['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']]).astype(int)
    L = data.loc[data['reaction_time'] <= q].copy()
    L_Y = create_Y(L)
    L_one_hot = np.array(L_Y[['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']]).astype(int)
    # run EM
    R_sem = run_em(R_one_hot, K=10, max_iter=150)
    L_sem = run_em(L_one_hot, K=10, max_iter=150)
    return R_sem, L_sem

# run EM for each quantile
R_sems = []
L_sems = []
for q in rt_quantiles:
    R_sem, L_sem = run_em_for_quantile(q, cifar10h_delta)
    R_sems.append(R_sem)
    L_sems.append(L_sem)

In [77]:
delta = []
delta_var = []
C = len(rt_quantiles)
for i in range(C):
    delta.append(R_sems[i][8] - L_sems[i][8])
    delta_var.append(R_sems[i][9] + L_sems[i][9])


In [99]:
statistics = []

def T_statistic(delta, delta_var):
    stat = delta @ np.linalg.inv(delta_var) @ delta
    return stat

# compute the T statistic for each delta
for i in range(C):
    stat = T_statistic(delta[i], delta_var[i])
    statistics.append(stat)

max_T_index = np.argmax(statistics)

# return the quantile with the maximum norm
max_T_quantile = rt_quantiles[max_T_index]

statistics

[498398.88231013535,
 441980.73485241825,
 534065.6415015273,
 666265.5103522043,
 773233.9432338091,
 732822.8020818373,
 623363.8409889416,
 507887.0001658579,
 271801.72287553496]

In [93]:
R_sem_final, L_sem_final = run_em_for_quantile(max_T_quantile, cifar10h_theta)

In [109]:
# compute the final delta
final_delta = R_sem_final[8] - L_sem_final[8]

# compute the variance of the final delta
final_delta_var = R_sem_final[9] + L_sem_final[9]

# compute the T statistic of the final delta
final_T = T_statistic(final_delta, final_delta_var)

# compute the p-value of the final T statistic
final_p = 1 - chi2.cdf(final_T, 90)

final_p

'0.00e+00'

Next we run the SEM algorithm on the CIFAR-10 dataset, denoted by $\bm{Y}$. The algorithms returns:

0. $\ell(\bm{Y}; \hat{\bm{\pi}}, \hat{\Theta})$
1. $\hat{\bm{\pi}}$
2. $\hat{\Theta}$
3. $\hat{\tau}$
4. $(\hat{\Theta}_t)_{t=1,\dots,T}$
5. $(\hat{\bm{\pi}}_t)_{t=1,\dots,T}$
6. $(\hat{\tau}_t^{(i)})_{t=1,\dots,T}$
7. $(Z_t^{(i)})_{t=1,\dots,T}$
8. $\hat{\bm{\vartheta}}$
9. $\widehat{\mathbb{V}}(\hat{\bm{\vartheta}})$

In [82]:
# sort the confusion matrix
confusion_matrix_R = sort_matrix(R_sem_final[2])
confusion_matrix_L = sort_matrix(L_sem_final[2])

In [83]:
# Convert theta_matched, pi_matched, and tau_matched to DataFrames
theta_R = pd.DataFrame(confusion_matrix_R)
pi_R = pd.DataFrame(R_sem_final[1])
tau_R = pd.DataFrame(R_sem_final[3])
theta_L = pd.DataFrame(confusion_matrix_L)
pi_L = pd.DataFrame(L_sem_final[1])
tau_L = pd.DataFrame(L_sem_final[3])


We compute the posterior entropy of the images using the posterior probabilities $\hat{\tau}$.
We define 
$$H^{(i)} = -\sum_{z=1}^{10} \hat{\tau}_z^{(i)} \log_2(\hat{\tau}_z^{(i)})$$

In [84]:
# compute the posterior entropy
post_entropy_R = pd.DataFrame(-np.sum(R_sem_final[3] * np.log2(R_sem_final[3]), axis=1))
post_entropy_L = pd.DataFrame(-np.sum(L_sem_final[3] * np.log2(L_sem_final[3]), axis=1))

Next, we will compute the varince of the estimated parameter $\hat{\Theta}$ and the vectorized version of the estimated parameter $\hat{\bm{\vartheta}}$.


In [85]:
theta_old_R = pd.DataFrame(R_sem_final[8])
theta_old_L = pd.DataFrame(L_sem_final[8])
variance_theta_R = pd.DataFrame(R_sem_final[9])
variance_theta_L = pd.DataFrame(L_sem_final[9])

In [86]:
# export the results to csv
theta_R.to_csv('../data/theta_R.csv', index=False)
pi_R.to_csv('../data/pi_R.csv', index=False)
tau_R.to_csv('../data/tau_R.csv', index=False)
theta_L.to_csv('../data/theta_L.csv', index=False)
pi_L.to_csv('../data/pi_L.csv', index=False)
tau_L.to_csv('../data/tau_L.csv', index=False)
post_entropy_R.to_csv('../data/post_entropy_R.csv', index=False)
post_entropy_L.to_csv('../data/post_entropy_L.csv', index=False)
theta_old_R.to_csv('../data/theta_old_R.csv', index=False)
theta_old_L.to_csv('../data/theta_old_L.csv', index=False)
variance_theta_R.to_csv('../data/variance_theta_R.csv', index=False)
variance_theta_L.to_csv('../data/variance_theta_L.csv', index=False)